# Data and Packages

In [29]:
# Main Packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import scipy

# Clustering 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, adjusted_rand_score

# Parallel processing 
from joblib import Parallel, delayed

In [5]:
# Time constants
seconds_in_day = 60 * 60 * 24 
minutes_per_week = 7 * 24 * 60 
n_weeks = 8     
eight_seconds_week = n_weeks * minutes_per_week * 60

In [6]:
# Range of k values
k_range = range(2, 9)

In [7]:
# Load data and filter for human users & first 8 weeks of data
df = (
    pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz", has_header=False, separator=",",
                new_columns=['time','src_user','dest_user','src_comp','dest_comp',
                              'auth_type','logon_type','auth_orientation','outcome'])
    .filter(pl.col('src_user').str.starts_with('U'))
    .filter(pl.col('time') < eight_seconds_week)
    .collect(engine='streaming')
)

In [8]:
# Chosen features
feature_cols = [
    'log_n_events',
    'log_n_distinct_src',
    'log_n_distinct_dest',
    'failure_ratio',
    'c_bar',
    's_bar',
    ]

# Functions

In [9]:
# Build the features dataframe
def build_features(df, agg_minutes):

    agg_seconds = agg_minutes * 60

    return (
        df.lazy()
        .with_columns(
            bucket = pl.col('time') // agg_seconds,
            theta = ((pl.col('time') % seconds_in_day) / seconds_in_day) * 2 * np.pi,
            is_failure = (pl.col('outcome') == 'Fail').cast(pl.Int8),
        )
        .group_by(['src_user', 'bucket'])
        .agg(
            n_events = pl.len(),
            failure_ratio = pl.col('is_failure').mean(),
            n_distinct_src = pl.col('src_comp').n_unique(),
            n_distinct_dest = pl.col('dest_comp').n_unique(),
            c_bar = pl.col('theta').cos().mean(),
            s_bar = pl.col('theta').sin().mean(),
        )
        .with_columns(
            log_n_events = pl.col('n_events').log(),
            log_n_distinct_src = pl.col('n_distinct_src').log(),
            log_n_distinct_dest = pl.col('n_distinct_dest').log(),
        ).collect()
        )

In [10]:
def cluster_preprocess(features_df, X_scaled, week):

    lb = (week - 1) * buckets_per_week
    ub = lb + buckets_per_week - 1

    in_bin = features_df['bucket'].is_between(lb,ub).to_numpy()

    features_week = features_df.filter(in_bin)
    X_scaled_week = X_scaled[in_bin]

    return features_week, X_scaled_week 

In [11]:
def fit_kmeans(features_df, X_scaled, week, k_range):

    features_week, X_scaled_week = cluster_preprocess(features_df, X_scaled, week)

    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(X_scaled_week)

    features_week = (
        features_week.with_columns(pl.Series('cluster', labels))
        .select(['src_user', 'bucket', 'cluster'])
    )

    return week, features_week

In [12]:
def fit_kmeans(features_df, X_scaled, week, k_range):

    features_week, X_scaled_week = cluster_preprocess(features_df, X_scaled, week)

    ch_scores = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=123, n_init=10)
        labels = km.fit_predict(X_scaled_week)
        ch_scores[k] = calinski_harabasz_score(X_scaled_week, labels)

    best_k = max(ch_scores, key=ch_scores.get)

    return week, ch_scores, best_k

In [13]:
def ch_by_week(df, agg_hours, k_range):

    agg_minutes = round(agg_hours * 60)

    global buckets_per_week
    buckets_per_week = minutes_per_week // agg_minutes

    # Build features
    features_df = build_features(df, agg_minutes)

    # Standardise features
    X = features_df.select(feature_cols).to_numpy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Fit KMeans across k_range for each week in parallel
    weekly_results = Parallel(n_jobs=4)(
        delayed(fit_kmeans)(features_df, X_scaled, week, k_range) for week in range(1, n_weeks + 1)
    )

    ch_scores_by_week = {}
    best_k_by_week = {}

    for week, ch_scores, best_k in weekly_results:
        ch_scores_by_week[week] = ch_scores
        best_k_by_week[week] = best_k

    return ch_scores_by_week, best_k_by_week

# Cluster Sizes

In [14]:
def best_k_across_levels(df, agg_levels, k_range=k_range):

    rows = []

    for agg_hours in agg_levels:
        _, best_k_by_week = ch_by_week(df, agg_hours, k_range)

        for week, best_k in best_k_by_week.items():
            rows.append({'agg_hours': agg_hours, 'week': week, 'best_k': best_k})

    return pl.DataFrame(rows)

In [15]:
agg_levels = [1/60, 1/6, 0.5, 1, 2, 3, 4, 6, 8, 12, 24]  

results = best_k_across_levels(df, agg_levels)

# best k per week, one column per aggregation level
k_table = results.pivot(on='agg_hours', index='week', values='best_k')

In [ ]:
k_table

week,0.016666666666666666,0.16666666666666666,0.5,1.0,2.0,3.0,4.0,6.0,8.0,12.0,24.0
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,2,2,2,2,6,2,2,2,5,2,2
2,2,2,2,2,6,2,2,6,5,2,2
3,2,2,2,2,6,2,2,6,5,2,2
4,2,2,2,2,6,2,2,6,5,2,2
5,2,2,2,2,6,2,2,2,5,2,2
6,5,2,2,2,2,2,2,2,2,2,2
7,3,3,3,3,2,2,2,2,2,2,2
8,3,3,3,3,2,2,2,2,2,2,2


In [28]:
values, counts = np.unique(k_table.to_numpy(), return_counts=True)

for v, c in zip(values, counts):
    print(f"Cluster size: {v} -> Count: {c}")

Cluster size: 1 -> Count: 1
Cluster size: 2 -> Count: 67
Cluster size: 3 -> Count: 9
Cluster size: 4 -> Count: 1
Cluster size: 5 -> Count: 7
Cluster size: 6 -> Count: 9
Cluster size: 7 -> Count: 1
Cluster size: 8 -> Count: 1
